<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/05_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Crop labels (SPAM 2020)

**Purpose:** Build the crop labels the model learns from: which crops are grown where, per SPAM pixel, and a map from every 2 km grid cell to its pixel. Independent ground-truth points for validation are in notebook 05b.

**Inputs:**
- SPAM 2020 v2.2 global CSV files (IFPRI; doi 10.7910/DVN/SWPENT; mapspam.info): harvested area, physical area and yield, for all technologies (TA), irrigated (TI) and rainfed (TR)
- `grid.parquet` from notebook 01

**Method:**
1. The global pixel tables are read in chunks, filtered to the study region (-19 to 26 E, 3 to 19 N) and saved as small parquet files.
2. Each grid cell is matched, by integer pixel index, to the ~9 km (5 arc-minute) SPAM pixel containing it.
3. Per pixel, each crop's share of total harvested area is computed. A crop counts as "present" at a share of at least 5%.
4. Crops kept: share of at least 5% in at least 300 matched pixels, excluding generic groups (other cereals, pulses, oil crops, roots, fibre crops, vegetables, tropical and temperate fruit, rest of crops) and humid-zone tree crops (cocoa, plantain, rubber, oil palm, coffee, banana, tea, coconut, citrus). Result: 16 crops.

**Outputs** (Google Drive, not in this repo):
- `labels_pixel.parquet`: one row per SPAM pixel (21,128) with total harvested area, irrigated share, and area, share and presence for each of the 16 crops, plus the number of grid cells in the pixel
- `grid_pixel_map.parquet`: maps each of the 301,142 grid cells to its pixel; `has_label` is False for 0.26% of cells

**Notes:**
- Labels are at ~9 km and the grid is 2 km, so about 14 grid cells share each pixel. Weight model rows by 1 / cells-per-pixel and keep all cells of a pixel in the same cross-validation fold.
- Labels are concentrated: six crops (sorghum, pearl millet, maize, cowpea, groundnut, rice) dominate 82.8% of pixels. Sorghum and pearl millet are present at the 5% level in 82% and 69% of pixels, so share is the main target for them. Sugarcane, soybean, tomato and sweet potato have under 520 positive pixels each.
- SPAM is a modelled product that downscales subnational statistics, using Global Agro-Ecological Zones v4 among its inputs, so part of the signal may reflect GAEZ's own logic. Ground-truth points (05b) provide an independent check.
- Grid cells with no SPAM pixel (0.26%, mostly Mauritania and Senegal) have no label.
- Date palm is not a SPAM crop. Units are not stated in the CSV files (area presumed hectares; yield kg/ha per the SPAM README).

In [ ]:
# @title Cell 1: setup
import os, subprocess, zipfile
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
RAW = f'{BASE}/crop_project/raw/spam2020'
os.makedirs(RAW, exist_ok=True)

grid = pd.read_parquet(f'{BASE}/crop_project/grid.parquet')
BBOX = (-19.0, 3.0, 26.0, 19.0)      # lon_min, lat_min, lon_max, lat_max (a margin around the grid)
print(grid.shape)

In [ ]:
# @title Cell 2: get the zip files into Drive
URLS = {'harvested': '', 'physical': '', 'yield': ''}      # paste the three CSV link addresses

for name, url in URLS.items():
    out = f'{RAW}/spam2020_{name}_csv.zip'
    if url and not os.path.exists(out):
        subprocess.run(['wget', '-q', '-O', out, url], check=False)
    if os.path.exists(out):
        print(f'{name}: found, {os.path.getsize(out) / 1e6:.0f} MB')
    else:
        print(f'{name}: MISSING')

In [ ]:
# @title Cell 3: look inside before writing the filter
for name in URLS:
    z = f'{RAW}/spam2020_{name}_csv.zip'
    if os.path.exists(z):
        with zipfile.ZipFile(z) as zf:
            print(name, [(i.filename, round(i.file_size / 1e6)) for i in zf.infolist()])

with zipfile.ZipFile(f'{RAW}/spam2020_harvested_csv.zip') as zf:
    ta = [n for n in zf.namelist() if n.endswith('_TA.csv')][0]
    with zf.open(ta) as f:
        head = pd.read_csv(f, nrows=5)

print('\nFile:', ta)
print('Columns:', list(head.columns))
print(head.filter(regex='(?i)unit|year|tech|rec|adm0|^x$|^y$').to_string())

In [ ]:
# @title Cell 4: filter SPAM to the study region
# Reads each global SPAM CSV in chunks, keeps pixels inside BBOX, saves region parquet files,
# then checks how the 2 km grid cells match the 5 arc-minute SPAM pixels.
ID_COLS = ['grid_code', 'x', 'y', 'FIPS0', 'FIPS1', 'FIPS2', 'ADM0_NAME', 'ADM1_NAME', 'ADM2_NAME']
ZIPS = {'H': 'harvested', 'A': 'physical', 'Y': 'yield'}
KEYS = ['H_TA', 'H_TI', 'H_TR', 'A_TA', 'A_TI', 'A_TR', 'Y_TA', 'Y_TI', 'Y_TR']

def read_region(zip_path, key, bbox, chunksize=200_000):
    lon0, lat0, lon1, lat1 = bbox
    parts = []
    with zipfile.ZipFile(zip_path) as zf:
        member = [n for n in zf.namelist() if n.endswith(f'_{key}.csv')][0]
        with zf.open(member) as f:
            for ch in pd.read_csv(f, chunksize=chunksize):
                sel = ch[(ch.x >= lon0) & (ch.x <= lon1) & (ch.y >= lat0) & (ch.y <= lat1)]
                if len(sel):
                    parts.append(sel)
    return pd.concat(parts, ignore_index=True)

spam = {}
for key in KEYS:
    path = f'{RAW}/spam2020_region_{key}.parquet'
    if os.path.exists(path):
        spam[key] = pd.read_parquet(path)
    else:
        d = read_region(f'{RAW}/spam2020_{ZIPS[key[0]]}_csv.zip', key, BBOX)
        crops = [c for c in d.columns if c not in ID_COLS]
        d[crops] = d[crops].astype('float32')
        d.to_parquet(path, index=False)
        spam[key] = d
    print(key, spam[key].shape)

h = spam['H_TA']
crops = [c for c in h.columns if c not in ID_COLS]
print('\nPixels in the region:', len(h))
print(h.ADM0_NAME.value_counts().head(15))
print('\nTop crops by total harvested area in the region (file units):')
print(h[crops].sum().sort_values(ascending=False).head(10).round(0))

# Are pixel centres on a regular 5 arc-minute grid? (deviation should be ~0)
frac = (h.x + 180) * 12 - 0.5
print('\nPixel centre deviation (max):', float(np.abs(frac - np.round(frac)).max()))

# How many of our grid cells fall inside a SPAM pixel that is present in the table?
def pixel_index(lon, lat):
    return (np.floor((np.asarray(lon, dtype='float64') + 180) * 12).astype(int),
            np.floor((90 - np.asarray(lat, dtype='float64')) * 12).astype(int))

gp = grid[['grid_id', 'adm0', 'lon', 'lat']].copy()
gp['px'], gp['py'] = pixel_index(gp.lon, gp.lat)
sp = h[['x', 'y']].copy()
sp['px'], sp['py'] = pixel_index(sp.x, sp.y)
sp = sp[['px', 'py']].drop_duplicates().assign(in_spam=True)
m = gp.merge(sp, on=['px', 'py'], how='left')
print('\nShare of grid cells whose SPAM pixel is in the table (%):', round(float(m.in_spam.notna().mean() * 100), 2))
print((m.groupby('adm0').in_spam.apply(lambda s: s.notna().mean() * 100)).round(1).sort_values())
print('Distinct SPAM pixels matched:', m.dropna(subset=['in_spam'])[['px', 'py']].drop_duplicates().shape[0])

In [ ]:
# @title Cell 5: crop inventory
#@title Crop inventory over the matched SPAM pixels { display-mode: "both" }
# For the SPAM pixels that contain our grid cells: area, presence and dominance per crop.
matched = m.dropna(subset=['in_spam'])[['px', 'py']].drop_duplicates()
idx = pd.MultiIndex.from_frame(matched)

Hdf = h.copy()
Hdf['px'], Hdf['py'] = pixel_index(Hdf.x, Hdf.y)
HA = Hdf.set_index(['px', 'py'])[crops].reindex(idx)
print('Matched pixels:', len(HA), '| with zero total harvested area:', int((HA.sum(axis=1) == 0).sum()))

tot = HA.sum(axis=1)
share = HA.div(tot.replace(0, np.nan), axis=0)
dom = share.idxmax(axis=1).value_counts()

inv = pd.DataFrame({
    'total_area': HA.sum(),
    'pixels_present': (HA > 0).sum(),
    'pixels_share_ge_5pct': (share >= 0.05).sum(),
    'pixels_dominant': dom}).fillna(0)
inv['pct_pixels_present'] = (inv.pixels_present / len(HA) * 100).round(1)
inv = inv.sort_values('total_area', ascending=False)
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 60)
print(inv.round(0).astype({'pixels_present': int, 'pixels_share_ge_5pct': int, 'pixels_dominant': int}))

n_dom = int((inv.pixels_dominant / len(HA) >= 0.01).sum())
top6 = dom.head(6)
print(f'\nCrops that are dominant in at least 1% of pixels: {n_dom}')
print(f'Top 6 dominant crops cover {top6.sum() / dom.sum() * 100:.1f}% of pixels:', list(top6.index))

In [ ]:
# @title Cell 6: build the labels
#@title Build SPAM labels for the grid { display-mode: "both" }
MIN_PIXELS = 300       # keep crops with share >= SHARE_MIN in at least this many matched pixels
SHARE_MIN  = 0.05      # "present" = crop is at least 5% of the pixel's harvested area
EXCLUDE = ['ocer', 'opul', 'ooil', 'rest', 'trof', 'temf', 'ofib', 'orts', 'vege',      # generic groups
           'coco', 'plnt', 'rubb', 'oilp', 'coff', 'rcof', 'bana', 'teas', 'cnut', 'citr']  # humid-zone tree crops
CROPS = [c for c in inv.index if c not in EXCLUDE and inv.loc[c, 'pixels_share_ge_5pct'] >= MIN_PIXELS]
print(len(CROPS), 'crops:', CROPS)

# Irrigated fraction per pixel (irrigated harvested area / total harvested area)
HI = spam['H_TI'].copy()
HI['px'], HI['py'] = pixel_index(HI.x, HI.y)
irrig = HI.set_index(['px', 'py'])[crops].sum(axis=1).reindex(idx).fillna(0)

lab = pd.DataFrame(index=idx)
lab['total_area'] = tot.values
lab['irrig_share'] = (irrig.values / tot.values).clip(0, 1)
for c in CROPS:
    lab[f'area_{c}'] = HA[c].values
    lab[f'share_{c}'] = share[c].values.astype('float32')
    lab[f'present_{c}'] = (share[c].values >= SHARE_MIN)

# Map every grid cell to its pixel; count cells per pixel
cell_map = m[['grid_id', 'px', 'py', 'in_spam']].copy()
cell_map['has_label'] = cell_map.in_spam.notna()
cell_map = cell_map.drop(columns='in_spam')
lab = lab.join(cell_map[cell_map.has_label].groupby(['px', 'py']).size().rename('n_cells'))

lab.reset_index().to_parquet(f'{BASE}/crop_project/labels_pixel.parquet', index=False)
cell_map.to_parquet(f'{BASE}/crop_project/grid_pixel_map.parquet', index=False)
print('labels_pixel:', lab.shape, '| grid_pixel_map:', cell_map.shape)

# Summary: how much signal does each crop have?
pres = lab[[f'present_{c}' for c in CROPS]]
summ = pd.DataFrame({'pixels_present_5pct': pres.sum().values,
                     'pct_pixels': (pres.mean() * 100).round(1).values,
                     'mean_share': [lab[f'share_{c}'].mean() for c in CROPS]}, index=CROPS)
print(summ.round(3))
print('\nCrops present (>=5%) per pixel:'); print(pres.sum(axis=1).describe().round(2))
print('Irrigated share: median', round(float(lab.irrig_share.median()), 3),
      '| pixels with > 20% irrigated:', int((lab.irrig_share > 0.2).sum()))

from google.colab import drive      # commit the new files to Drive
drive.flush_and_unmount(); drive.mount('/content/drive')